# CIS6005 Computational Intelligence
## Notebook 11 — Kaggle Submission
**Student Health Risk Prediction | Kaggle PS S6E7**

---
### Process
1. Load the champion model
2. Load preprocessed test set (from Phase 6)
3. Generate predictions on test data
4. Format as required by Kaggle: `id, health_condition`
5. Save CSV and upload to Kaggle

> **Kaggle requires:** A CSV with exactly two columns: `id` and `health_condition`

In [1]:
# ============================================================
# IMPORTS & SETUP
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROC_DATA    = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR   = PROJECT_ROOT / 'models'
SUBMIT_DIR   = PROJECT_ROOT / 'data' / 'submissions'

X_test        = np.load(PROC_DATA / 'X_test.npy')
test_ids      = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'test.csv')['id']
label_encoder = joblib.load(MODELS_DIR / 'label_encoder.joblib')
CLASS_NAMES   = list(label_encoder.classes_)

print(f'Test set shape : {X_test.shape}')
print(f'Test IDs count : {len(test_ids)}')
print(f'Classes        : {CLASS_NAMES}')

Test set shape : (295753, 19)
Test IDs count : 295753
Classes        : ['at-risk', 'fit', 'unhealthy']


## 1. Load Champion Model

In [2]:
# ============================================================
# SECTION 1: Load Champion Model
# ============================================================

if (MODELS_DIR / 'model_tuned_best.joblib').exists():
    champion      = joblib.load(MODELS_DIR / 'model_tuned_best.joblib')
    champion_name = 'Tuned Best Model'
elif (MODELS_DIR / 'model_random_forest.joblib').exists():
    champion      = joblib.load(MODELS_DIR / 'model_random_forest.joblib')
    champion_name = 'Random Forest'
else:
    raise FileNotFoundError('No model found. Run Notebooks 07 and 09 first.')

print(f'Champion model  : {champion_name}')
print(f'Algorithm type  : {type(champion).__name__}')

Champion model  : Tuned Best Model
Algorithm type  : RandomForestClassifier


## 2. Generate Predictions on Test Set

In [3]:
# ============================================================
# SECTION 2: Generate Test Predictions
# ============================================================

y_test_encoded = champion.predict(X_test)
y_test_labels  = label_encoder.inverse_transform(y_test_encoded)

print('=' * 50)
print('  Test Set Prediction Distribution')
print('=' * 50)
unique, counts = np.unique(y_test_labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    pct = cnt / len(y_test_labels) * 100
    print(f'  {cls:<15}: {cnt:>6,}  ({pct:.1f}%)')
print(f'  {"Total":<15}: {len(y_test_labels):>6,}')
print('=' * 50)

  Test Set Prediction Distribution
  at-risk        : 261,593  (88.4%)
  fit            : 14,073  (4.8%)
  unhealthy      : 20,087  (6.8%)
  Total          : 295,753


## 3. Create and Validate Submission File

In [4]:
# ============================================================
# SECTION 3: Build + Validate Submission DataFrame
# ============================================================

submission_df = pd.DataFrame({
    'id'              : test_ids.values,
    'health_condition': y_test_labels
})

sample_sub = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'sample_submission.csv')

# Validation checks
assert len(submission_df) == len(sample_sub), \
    f'Row mismatch: got {len(submission_df)}, expected {len(sample_sub)}'
assert list(submission_df.columns) == list(sample_sub.columns), \
    f'Column mismatch: {list(submission_df.columns)}'
assert submission_df['health_condition'].isin(CLASS_NAMES).all(), \
    'Invalid class labels in predictions'

print('Validation passed!')
print(f'  Rows    : {len(submission_df):,}')
print(f'  Columns : {list(submission_df.columns)}')
display(submission_df.head())

Validation passed!
  Rows    : 295,753
  Columns : ['id', 'health_condition']


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## 4. Save Submission CSV

In [5]:
# ============================================================
# SECTION 4: Save Submission File
# ============================================================

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
fname     = f'submission_{champion_name.replace(" ", "_")}_{timestamp}.csv'
out_path  = SUBMIT_DIR / fname

submission_df.to_csv(out_path, index=False)

print('=' * 60)
print('  PHASE 11 COMPLETE — Kaggle Submission Ready')
print('=' * 60)
print(f'  File : {fname}')
print(f'  Path : {out_path}')
print(f'  Rows : {len(submission_df):,}')
print('=' * 60)
print()
print('  UPLOAD STEPS:')
print('  1. Go to kaggle.com/competitions/playground-series-s6e7')
print('  2. Click Submit Predictions')
print('  3. Upload the file from:')
print(f'     {out_path}')
print('  4. Add a note and submit')
print('  5. Record your leaderboard score in your report')
print('=' * 60)

  PHASE 11 COMPLETE — Kaggle Submission Ready
  File : submission_Tuned_Best_Model_20260722_231532.csv
  Path : d:\Student_Health_Risk_Prediction\data\submissions\submission_Tuned_Best_Model_20260722_231532.csv
  Rows : 295,753

  UPLOAD STEPS:
  1. Go to kaggle.com/competitions/playground-series-s6e7
  2. Click Submit Predictions
  3. Upload the file from:
     d:\Student_Health_Risk_Prediction\data\submissions\submission_Tuned_Best_Model_20260722_231532.csv
  4. Add a note and submit
  5. Record your leaderboard score in your report
